### Universidad del Valle de Guatemala
### Facultad de Ingeniería — Departamento de Ciencias de la Computación
### CC3084 – Data Science | Semestre II – 2026

***Proyecto 1: Obtención y Limpieza de los Datos***

**Tema:** Establecimientos educativos de Guatemala — nivel escolar **Diversificado**
**Fuente de datos:** Sistema de Búsqueda de Establecimientos Educativos, MINEDUC Guatemala — http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/
**Grupo:** 2

## Integrantes
| Integrante | Carné |
|---|---|
| Jose Donado | 22684 |
| Daniela Ramírez | 23053 |
| Cindy Gualim | 21226 |

## Objetivo del proyecto
Acceder a los datos de establecimientos educativos de todo el país que lleguen hasta el nivel Diversificado, y limpiarlos de la forma más transparente y reproducible posible. Concretamente, este notebook documenta:
1. La obtención de los datos crudos (web scraping con Selenium) y su filtrado al nivel escolar exigido.
2. Un diagnóstico del estado inicial de los datos (registros, tipos, faltantes, únicos, duplicados, valores fuera de dominio, formatos inconsistentes).
3. Un plan de limpieza por variable (problema → regla de corrección → riesgos).
4. La limpieza del conjunto de datos completo, variable por variable.
5. Un registro de todas las transformaciones aplicadas.
6. Pruebas automáticas de validación del conjunto limpio.
7. Un informe de calidad que compara el estado antes vs. después de la limpieza.
8. La generación de un único conjunto de datos limpio, para todos los departamentos.
9. Un Libro de Códigos (`CODEBOOK.md`, en la raíz del repositorio) con los metadatos necesarios para interpretar el conjunto limpio.

---
### ***PARTE 0 — Obtención de datos (Web Scraping con Selenium)***

Se recorre el formulario público del MINEDUC (`cmbDepartamento`, `cmbMunicipio`, `cmbNivel`, `cmbSector`, `ddlplan`, `ddlModalidad`), consultando cada uno de los 22 departamentos en los niveles BASICO y DIVERSIFICADO, con Municipio/Sector/Plan/Modalidad = "TODOS". Cada resultado se extrae como tabla HTML y se convierte a `DataFrame` con pandas. Al final se concatenan todas las consultas y se guarda el crudo en `establecimientos.csv`.

> La consigna del proyecto solo exige el nivel **DIVERSIFICADO**; ese filtro se aplica al inicio de la Parte 1, dejando aquí el crudo completo tal como se extrajo (trazabilidad total).

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd
import time

def consultar(driver, departamento, nivel):

    wait = WebDriverWait(driver, 20)

    # Volver a la página principal
    driver.get("https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/")

    # Esperar que cargue el formulario
    wait.until(
        EC.presence_of_element_located(
            (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
        )
    )

    # Departamento
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )).select_by_visible_text(departamento)

    # Esperar que actualice el municipio
    time.sleep(2)

    # Municipio
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbMunicipio"
    )).select_by_visible_text("TODOS")

    # Nivel
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbNivel"
    )).select_by_visible_text(nivel)

    # Sector
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbSector"
    )).select_by_visible_text("TODOS")

    # Plan
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlplan"
    )).select_by_visible_text("TODOS")

    # Modalidad
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlModalidad"
    )).select_by_visible_text("TODOS")

    # Buscar
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:IbtnConsultar"
    ).click()

    # Esperar la tabla
    wait.until(
        EC.presence_of_element_located(
            (By.ID, "_ctl0_ContentPlaceHolder1_dgResultado")
        )
    )

    # Extraer HTML
    tabla = driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_dgResultado"
    )

    html = tabla.get_attribute("outerHTML")

    # Convertir a DataFrame
    df = pd.read_html(StringIO(html))[0]

    # Limpiar
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = df.drop(df.columns[0], axis=1)

    return df

In [ ]:
departamentos = [
    "GUATEMALA",
    "EL PROGRESO",
    "SACATEPEQUEZ",
    "CHIMALTENANGO",
    "ESCUINTLA",
    "SANTA ROSA",
    "SOLOLA",
    "TOTONICAPAN",
    "QUETZALTENANGO",
    "SUCHITEPEQUEZ",
    "RETALHULEU",
    "SAN MARCOS",
    "HUEHUETENANGO",
    "QUICHE",
    "BAJA VERAPAZ",
    "ALTA VERAPAZ",
    "PETEN",
    "IZABAL",
    "ZACAPA",
    "CHIQUIMULA",
    "JALAPA",
    "JUTIAPA"
]
niveles = [
    "BASICO",
    "DIVERSIFICADO"
]

todos = []

for depto in departamentos:

    for nivel in niveles:

        print(f"Consultando {depto} - {nivel}")

        df = consultar(driver, depto, nivel)

        todos.append(df)

In [ ]:
df_final = pd.concat(todos, ignore_index=True)

df_final.to_csv(
    "establecimientos.csv",
    index=False,
    encoding="utf-8-sig"
)

---
### ***PARTE 1 — Obtención y Diagnóstico***


**1.1 Carga de datos crudos y filtrado a NIVEL = DIVERSIFICADO (actividades 1-2)**

- Cargar `establecimientos.csv.xls` (salida del scraping de arriba).
- Filtrar `NIVEL == "DIVERSIFICADO"` (la consigna solo pide ese nivel).
- Guardar el crudo filtrado en `data/raw/establecimientos_diversificado_raw.csv`.

In [ ]:
#  NIVEL == "DIVERSIFICADO"
# y guardar en data/raw/establecimientos_diversificado_raw.csv


**1.2 Diagnóstico del estado inicial de los datos (actividad 3, a-h)**
Sobre el crudo ya filtrado a DIVERSIFICADO (9,706 registros). Debe incluir, respaldado con código:
- a. Número de registros y variables.
- b. Tipo de dato de cada variable.
- c. Cantidad y porcentaje de valores faltantes por variable.
- d. Cantidad de valores únicos por variable.
- e. Cantidad de registros duplicados exactos.
- f. Variables con valores fuera de dominio o inconsistentes.
- g. Variables con formatos inconsistentes (ej. `TELEFONO` con múltiples números, `DISTRITO` con 2 formatos distintos, comillas sueltas en `ESTABLECIMIENTO`).
- h. Resumen de problemas potenciales de calidad de datos detectados.

---
### ***PARTE 2 — Plan de Limpieza y Limpieza (variables de identificación, geografía y texto)***


**2.1 Plan de limpieza (actividad 4)**

Tabla por variable: problema detectado → regla de corrección (y por qué funcionará) → riesgos de aplicarla. Cubrir como mínimo: `CODIGO`, `DISTRITO`, `DEPARTAMENTO`, `MUNICIPIO`, `ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR`, `DIRECTOR`, `SECTOR`, `AREA`, `STATUS`, `MODALIDAD`, `JORNADA`, `PLAN`, `DEPARTAMENTAL`.

**2.2 Limpieza: renombrado de columnas y normalización general de texto (actividad 5a, 5c)**

- Renombrar columnas a snake_case descriptivo (actividad 9c).
- Normalizar texto en todas las columnas: strip, colapso de espacios múltiples, caracteres invisibles.
- Convertir placeholders de nulo ("N/A", "NULL", "-", ".", "Sin dato", "S/D", etc.) a NA.

**2.3 Limpieza: identificadores y geografía (actividad 5b, 5e, 5f)**

- `codigo_establecimiento`: validar formato `XX-XX-XXXX-XX` (llave primaria).
- `codigo_distrito`: documentar los formatos coexistentes; descartar códigos truncados.
- `departamento`: validar contra el catálogo de los 22 departamentos de Guatemala.
- `municipio`: mayúsculas consistentes; buscar variantes de escritura por similitud de cadenas dentro de cada departamento (sin corregir automáticamente homónimos reales).

**2.4 Limpieza: texto libre y categóricas (actividad 5c, 5d, 5f)**

- `nombre_establecimiento`: unificar comillas simples/dobles, quitar comillas sueltas al inicio/fin.
- `direccion`, `supervisor`, `director`: placeholders → NA (ya cubiertos en 2.2), revisar formato.
- `sector`, `estado`, `modalidad`, `jornada`, `plan_educativo`, `direccion_departamental`: validar dominio de categorías.
- `area`: tratar la categoría "SIN ESPECIFICAR" como valor faltante (análoga a "Sin dato").

---
### ***PARTE 3 — Teléfono, Duplicados, Validación, Informe y Cierre***


**3.1 Limpieza de `telefono` (actividad 5e, 5f, 5i)**

- Extraer números cuando el campo trae varios separados por guion/coma.
- Validar longitud de 8 dígitos (formato de teléfono en Guatemala).
- Variables derivadas: `telefono_valido` (bool), `telefono_multiple` (bool) — justificar por qué se crean y cómo se calculan (para el Libro de Códigos).

**3.2 Consistencia entre variables (actividad 5h)**

- Validar que `direccion_departamental` corresponda a `departamento` (incluyendo las subdivisiones de Guatemala: Norte/Sur/Oriente/Occidente, y de Quiché: Quiché/Quiché Norte).
- Documentar cualquier inconsistencia encontrada.

**3.3 Duplicados exactos y parciales (actividad 5g)**
- Duplicados exactos: detectar y eliminar (`drop_duplicates`).
- Duplicados parciales: similitud de cadenas (RapidFuzz) sobre nombre + dirección + municipio, cruzado con jornada/plan/sector para distinguir "misma sede, oferta distinta" (no es duplicado) de "posible duplicado real".
- **No eliminar automáticamente los duplicados parciales** — marcar con columnas derivadas (`posible_duplicado`, `grupo_posible_duplicado`) y documentar la decisión para cada caso/grupo.

**3.4 Registro de transformaciones (actividad 6)**

Tabla: `variable | problema_detectado | transformacion | registros_afectados | justificacion`, con una fila por cada cambio aplicado en la Parte 2 y la Parte 3.

**3.5 Validación automática del conjunto limpio (actividad 7)**

Comprobar programáticamente que:
- No existen registros duplicados exactos.
- No existen espacios al inicio/final de textos.
- Los teléfonos tienen formato consistente.
- Departamentos y municipios pertenecen al catálogo correspondiente.
- Las variables tienen el tipo de dato esperado.
- No existen categorías duplicadas por diferencias de escritura.
- No existen los valores inválidos detectados en el diagnóstico inicial.

**3.6 Informe de calidad: antes vs. después (actividad 8)**

Tabla con: Registros, Variables, Valores faltantes, Variables con NA, Duplicados exactos, Posibles duplicados, Variables con formato inconsistente, Variables con tipo incorrecto, Categorías inconsistentes, Errores corregidos.

**3.7 Generación del conjunto de datos limpio final (actividad 9)**

Exportar a `data/clean/establecimientos_diversificado_limpio.csv`: estructura consistente, tipos correctos, nombres descriptivos, formato uniforme, sin los errores detectados en la validación.

## 3.8 Libro de Códigos (actividad 10)
Ver `CODEBOOK.md` en la raíz del repositorio (y su versión .pdf para la entrega). Debe incluir, por variable: descripción, tipo de dato, dominio/valores posibles, tratamiento aplicado, variables derivadas, fecha de extracción, fuente y versión del conjunto limpio.

---
### Notas de equipo
- Cada integrante trabaja y comitea su(s) sección(es) de forma incremental — esto es lo que se revisa como "contribución significativa".
- Antes de correr la Parte 2/3, asegúrate de haber ejecutado la Parte 1 (carga del crudo) en la misma sesión del kernel.
- Cualquier decisión no trivial (ej. qué hacer con un duplicado parcial) se documenta en el registro de transformaciones (3.4), no solo en el código.